## Cell 1 — Clone or update the repo

Pulls fresh code on every notebook run. `cd` into the repo root so all relative paths (`CMaps/`, `stage3/`, `stage2_refactor/`) resolve.

In [ ]:
import os, subprocess

REPO_URL = "https://github.com/m8wei-coder/ECE-228-project"
REPO_DIR = "/content/ECE-228-project"

if os.path.isdir(REPO_DIR):
    print(f"repo exists at {REPO_DIR}, pulling...")
    subprocess.run(["git", "-C", REPO_DIR, "pull"], check=True)
else:
    print(f"cloning {REPO_URL} -> {REPO_DIR} ...")
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)

os.chdir(REPO_DIR)
print("cwd:", os.getcwd())
!ls -la


## Cell 2 — Install dependencies

Colab usually ships with torch / numpy / pandas / scipy / scikit-learn / joblib / pyyaml. We add `torch_geometric` (the dense modules used by `stage3.models.gnn_modules` don't require `torch_scatter` / `torch_sparse`).

In [ ]:
# Core PyG; DenseGCNConv works without torch_scatter / torch_sparse.
!pip install -q torch_geometric

# stage2_refactor extras that may not be on Colab by default.
!pip install -q pyyaml joblib

# Uncomment if you want WandB logging:
# !pip install -q wandb
print("deps installed")


## Cell 3 — Mount Google Drive

All run outputs (checkpoints, summaries, logs) land under `/content/drive/MyDrive/ece228_stage3/` so a Colab disconnect does not lose results. Graph `.npy` files stay in the repo (`stage3/artifacts/*.npy`).

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os
DRIVE_ROOT = "/content/drive/MyDrive/ece228_stage3"
os.makedirs(DRIVE_ROOT, exist_ok=True)
os.makedirs(f"{DRIVE_ROOT}/runs", exist_ok=True)
print("Drive root:", DRIVE_ROOT)


## Cell 4 — Environment self-check

Confirm torch / torch_geometric / our stage2_refactor + stage3 imports all resolve, and report GPU availability.

In [ ]:
import sys
if "/content/ECE-228-project" not in sys.path:
    sys.path.insert(0, "/content/ECE-228-project")

import torch
print("torch:           ", torch.__version__)
print("cuda available:  ", torch.cuda.is_available())
if torch.cuda.is_available():
    print("  device:        ", torch.cuda.get_device_name(0))

import torch_geometric
print("torch_geometric: ", torch_geometric.__version__)

from stage2_refactor.data.io import read_cmapss_table
from stage2_refactor.training.trainer import fit
from stage2_refactor.training.evaluator import rmse_score
from stage3.models.recurrent_gnn import RecurrentGNNFusion
from stage3.models.gnn_modules import GCN
from stage3.build_graph import build_adjacency
import stage3.train_stage3 as t3
print("stage2_refactor + stage3 imports OK")


## Cell 5 — Build adjacency matrices

Recompute the Pearson / physical / union graphs for all four subsets and write them to `stage3/artifacts/adj_{subset}_{method}.npy`. The repo already ships these files (≤4 KB each, whitelisted in `.gitignore`); this cell exists so a fresh clone can regenerate them and so we can swap the threshold for ablation.

In [ ]:
import subprocess, sys

PEARSON_THRESHOLD = 0.3

result = subprocess.run(
    [sys.executable, "-m", "stage3.build_graph",
     "--threshold", str(PEARSON_THRESHOLD)],
    cwd="/content/ECE-228-project",
    capture_output=True, text=True,
)
print(result.stdout)
if result.returncode != 0:
    print("STDERR:\n", result.stderr)
    raise RuntimeError("build_graph failed")


## Cell 6 — Hyperparameters (the only knob cell)

Edit this cell only. Setting any value to `None` falls back to the per-subset default baked into `stage3/train_stage3.py:SUBSET_CONFIGS` (which mirrors Stage 2 finalists).

Per-subset defaults:
- FD001 / FD003 → GRU,  h=90, L=2, dropout=0.2, lr=5e-4
- FD002 / FD004 → BiGRU, h=60, L=2, dropout=0.1, lr=5e-4

In [ ]:
SUBSET           = "FD001"     # FD001 / FD002 / FD003 / FD004
GRAPH_METHOD     = "physical"  # physical / pearson / union
USE_GNN          = True         # False = pure recurrent ablation control

# Backbone overrides (None = use per-subset default).
RECURRENT_KIND   = None         # gru / bigru
HIDDEN_SIZE      = None
NUM_LAYERS       = None
DROPOUT          = None
LR               = None
BATCH_SIZE       = None
EPOCHS           = None         # default uses 150 (FD001/2/4) or 250 (FD003)

# GNN branch hyperparams.
GNN_HIDDEN       = 32
GNN_LAYERS       = 2
GNN_KIND         = "gcn"        # gcn (gat is reserved for later ablation)
GNN_DROPOUT      = 0.1
GNN_POOL         = "mean"

# Run id + seed.
SEED             = 42
RUN_ID           = None         # None = f"{subset.lower()}_seed{seed}"


## Cell 7 — Train

Reads variables from Cell 6, calls `stage3.train_stage3.run()` with checkpoints + summary going to Drive. The repo-local `stage3/artifacts/` is still used as the graph source (it holds the `.npy` adjacencies).

In [ ]:
import sys, importlib
import stage3.train_stage3 as t3
importlib.reload(t3)

RUN_ID_EFFECTIVE = RUN_ID or f"{SUBSET.lower()}_seed{SEED}"
OUTPUT_DIR       = f"{DRIVE_ROOT}/runs/{SUBSET}/{RUN_ID_EFFECTIVE}"
CHECKPOINT_DIR   = f"{OUTPUT_DIR}/checkpoints"

argv = [
    "--subset",       SUBSET,
    "--graph-method", GRAPH_METHOD,
    "--seed",         str(SEED),
    "--gnn-hidden",   str(GNN_HIDDEN),
    "--gnn-layers",   str(GNN_LAYERS),
    "--gnn-kind",     GNN_KIND,
    "--gnn-dropout",  str(GNN_DROPOUT),
    "--gnn-pool",     GNN_POOL,
    "--run-id",       RUN_ID_EFFECTIVE,
    "--output-dir",   OUTPUT_DIR,
    "--checkpoint-dir", CHECKPOINT_DIR,
]
argv += ["--use-gnn"] if USE_GNN else ["--no-gnn"]

for flag, val in [
    ("--recurrent-kind", RECURRENT_KIND), ("--hidden-size", HIDDEN_SIZE),
    ("--num-layers", NUM_LAYERS), ("--dropout", DROPOUT),
    ("--learning-rate", LR), ("--batch-size", BATCH_SIZE),
    ("--epochs", EPOCHS),
]:
    if val is not None:
        argv += [flag, str(val)]

print("argv:", argv)
saved_argv = sys.argv
sys.argv = ["train_stage3"] + argv
try:
    args = t3.parse_args()
    summary = t3.run(args)
finally:
    sys.argv = saved_argv

print(f"\n>>> test_rmse={summary['test_rmse']:.4f}  test_score={summary['test_score']:.4f}")


## Cell 8 — Evaluate / inspect the run

Reads `summary.json` + `train_log.csv` from the run directory in Drive and prints a tidy report. Re-run after Cell 7 to inspect that specific run.

In [ ]:
import json
import pandas as pd
from pathlib import Path

run_dir = Path(OUTPUT_DIR)
summary = json.loads((run_dir / "summary.json").read_text())

print("=== run summary ===")
print(f"subset:           {summary['subset']}")
print(f"run_id:           {summary['run_id']}")
print(f"seed:             {summary['seed']}")
print(f"use_gnn:          {summary['use_gnn']}")
print(f"graph_method:     {summary.get('graph_method')}")
print(f"backbone:         {summary['recurrent_kind']} (h={summary['hidden_size']}, L={summary['num_layers']}, d={summary['dropout']})")
print(f"best_epoch / val: {summary['best_epoch']}  (val_rmse={summary['best_metric']:.4f})")
print(f"test_rmse:        {summary['test_rmse']:.4f}")
print(f"test_score:       {summary['test_score']:.4f}")
print(f"train_sequences:  {summary['train_sequences']}")
print(f"val_sequences:    {summary['val_sequences']}")
print(f"parameter_count:  {summary['parameter_count']}")
print(f"train_seconds:    {summary['total_train_seconds']:.1f}")

log_path = run_dir / "train_log.csv"
if log_path.exists():
    log = pd.read_csv(log_path)
    cols = [c for c in ["epoch", "train_loss", "train_rmse", "val_loss", "val_rmse", "val_score"] if c in log.columns]
    print("\n=== last 5 epochs ===")
    print(log[cols].tail(5).to_string(index=False))


## Cell 9 (optional) — Batch: all 4 subsets × {GNN on, GNN off}

8 runs in sequence. Each run lands under `DRIVE_ROOT/runs/{SUBSET}/{run_id}/`. Results aggregated and written to `DRIVE_ROOT/batch_summary.json`. Adjust `BATCH_SEED` to repeat for multi-seed averages.

In [ ]:
import json, sys, importlib
from pathlib import Path
import stage3.train_stage3 as t3
importlib.reload(t3)

BATCH_SEED      = 42
BATCH_GRAPH     = "physical"
BATCH_EPOCHS    = {"FD001": 150, "FD002": 150, "FD003": 250, "FD004": 150}

results = []
for subset in ["FD001", "FD002", "FD003", "FD004"]:
    for use_gnn in [True, False]:
        tag      = "gnn" if use_gnn else "nognn"
        run_id   = f"{subset.lower()}_{tag}_seed{BATCH_SEED}"
        out_dir  = f"{DRIVE_ROOT}/runs/{subset}/{run_id}"
        ckpt_dir = f"{out_dir}/checkpoints"

        argv = [
            "--subset", subset, "--graph-method", BATCH_GRAPH,
            "--seed", str(BATCH_SEED), "--epochs", str(BATCH_EPOCHS[subset]),
            "--run-id", run_id,
            "--output-dir", out_dir, "--checkpoint-dir", ckpt_dir,
        ]
        argv += ["--use-gnn"] if use_gnn else ["--no-gnn"]

        print(f"\n=== {subset}  use_gnn={use_gnn}  run_id={run_id} ===")
        saved_argv = sys.argv
        sys.argv = ["train_stage3"] + argv
        try:
            args = t3.parse_args()
            s    = t3.run(args)
            results.append({
                "subset": subset, "use_gnn": use_gnn, "seed": BATCH_SEED,
                "test_rmse": s["test_rmse"], "test_score": s["test_score"],
                "best_epoch": s["best_epoch"], "params": s["parameter_count"],
                "train_seconds": s["total_train_seconds"],
            })
        finally:
            sys.argv = saved_argv

print("\n" + "=" * 70)
print("BATCH SUMMARY")
print("=" * 70)
print(f"{'subset':<8}{'use_gnn':<9}{'test_rmse':>12}{'test_score':>14}{'best_epoch':>13}{'params':>10}")
print("-" * 66)
for r in results:
    print(f"{r['subset']:<8}{str(r['use_gnn']):<9}{r['test_rmse']:>12.4f}{r['test_score']:>14.4f}{r['best_epoch']:>13d}{r['params']:>10d}")

summary_path = Path(DRIVE_ROOT) / "batch_summary.json"
summary_path.write_text(json.dumps(results, indent=2))
print(f"\nSaved: {summary_path}")
